In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e12/sample_submission.csv
/kaggle/input/playground-series-s5e12/train.csv
/kaggle/input/playground-series-s5e12/test.csv


In [2]:
class CONFIG:
    INPUT_DIR = '/kaggle/input/playground-series-s5e12'
    
    N_FOLDS = 5
    SEED = 42

    TARGET = 'diagnosed_diabetes'

config = CONFIG()

train = pd.read_csv(f'{config.INPUT_DIR}/train.csv')
test = pd.read_csv(f'{config.INPUT_DIR}/test.csv')
submission = pd.read_csv(f'{config.INPUT_DIR}/sample_submission.csv')

In [3]:
def eda(df, name):
    print(f"Exploring {name} dataframe")
    print('='*30)
    print(f'\n NULL VALUES: {df.isnull().sum()}')
    print('='*30)
    print(f'\n {name} dataframe shape: {df.shape}')
    print('='*30)
    print(f'\n {name} dataframe numerical stats: {df.describe()}')
    print('='*30)

eda(train, 'TRAIN')
eda(test, 'TEST')

Exploring TRAIN dataframe

 NULL VALUES: id                                    0
age                                   0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                     0
hdl_cholesterol                       0
ldl_cholesterol                       0
triglycerides                         0
gender                                0
ethnicity                             0
education_level                       0
income_level                          0
smoking_status                        0
employment_status                     0
family_history_diabetes               0
hypertension_history                  0

In [4]:
train['diagnosed_diabetes']

0         1.0
1         1.0
2         0.0
3         1.0
4         1.0
         ... 
699995    0.0
699996    1.0
699997    1.0
699998    1.0
699999    1.0
Name: diagnosed_diabetes, Length: 700000, dtype: float64

In [5]:
FEATURES = [col for col in train.columns if col not in ['id', 'diagnosed_diabetes']]
CATS = train[FEATURES].select_dtypes(include='object').columns

X = train[FEATURES].copy()
y = train[config.TARGET]

test_n = test[FEATURES].copy()

X[CATS] = X[CATS].astype('category')
test_n[CATS] = test_n[CATS].astype('category')

In [6]:
for c in CATS:
    for df in [X, test_n]:
        df[c], _ =  df[c].factorize()

In [7]:
from xgboost import XGBClassifier
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    # 'enable_categorical': True,
}

skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBClassifier(**params)

    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             verbose=1000)

    val_preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_preds

    fold_score = roc_auc_score(y_val, val_preds)
    print(f'FOLD {fold} AUC: {fold_score:.4f}')
    test_preds +=  model.predict_proba(test_n)[:, 1] / config.N_FOLDS

overall_auc = roc_auc_score(y, oof_preds)
print('='*30)
print(f"Overall OOF AUC: {overall_auc:.4f}")
print('='*30)

[0]	validation_0-auc:0.65460
[1000]	validation_0-auc:0.71922
[2000]	validation_0-auc:0.72405
[3000]	validation_0-auc:0.72585
[4000]	validation_0-auc:0.72690
[5000]	validation_0-auc:0.72741
[6000]	validation_0-auc:0.72779
[7000]	validation_0-auc:0.72803
[7797]	validation_0-auc:0.72814
FOLD 0 AUC: 0.7281
[0]	validation_0-auc:0.65528
[1000]	validation_0-auc:0.71714
[2000]	validation_0-auc:0.72171
[3000]	validation_0-auc:0.72362
[4000]	validation_0-auc:0.72470
[5000]	validation_0-auc:0.72538
[6000]	validation_0-auc:0.72579
[7000]	validation_0-auc:0.72603
[8000]	validation_0-auc:0.72622
[8373]	validation_0-auc:0.72626
FOLD 1 AUC: 0.7263
[0]	validation_0-auc:0.65569
[1000]	validation_0-auc:0.71769
[2000]	validation_0-auc:0.72246
[3000]	validation_0-auc:0.72458
[4000]	validation_0-auc:0.72571
[5000]	validation_0-auc:0.72631
[6000]	validation_0-auc:0.72671
[7000]	validation_0-auc:0.72697
[8000]	validation_0-auc:0.72714
[8664]	validation_0-auc:0.72723
FOLD 2 AUC: 0.7272
[0]	validation_0-auc:0.6

In [8]:
submission[config.TARGET] = test_preds
submission.to_csv(f'submission_cv_{overall_auc}.csv', index=False)